# EchoPilot Research 02: Dense Vector Indexing & Retrieval

This notebook demonstrates the local semantic search engine:
1. Using **`sentence-transformers/all-MiniLM-L6-v2`** to create 384-dimensional dense vectors.
2. Indexing document chunks in memory with cosine similarity.
3. Enforcing strict material isolation (retrieving chunks only from the target study material).
4. Evaluating retrieval accuracy using **Recall@K** and **MRR@K** (Mean Reciprocal Rank).


In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd

sys.path.append(os.path.abspath("../src"))
from retriever import MaterialVectorStore

print("VectorStore initialized.")


## Step 1: Load Chunks and Initialize Vector Store
We load the chunks generated from Notebook 01 and embed them locally.


In [ ]:
chunks_file = "../data/extracted_chunks.jsonl"
chunks = []
if os.path.exists(chunks_file):
    with open(chunks_file, "r", encoding="utf-8") as f:
        chunks = [json.loads(line) for line in f]
    print(f"Loaded {len(chunks)} chunks.")
else:
    print("No chunks file found. Generating fallback demonstration chunks...")
    chunks = [
        {"chunk_id": "deadlock_p4_c1", "material_id": "deadlock_demo", "material_name": "Deadlock.pdf", "page_number": 4, "section_heading": "Necessary conditions", "text": "Mutual Exclusion: A resource can only be shared in mutually exclusive manner so that two processes cannot use the same resource simultaneously.", "word_count": 24},
        {"chunk_id": "deadlock_p11_c7", "material_id": "deadlock_demo", "material_name": "Deadlock.pdf", "page_number": 11, "section_heading": "Banker's Algorithm", "text": "Need matrix is computed as Need[i, j] = Max[i, j] - Allocation[i, j]. This represents the remaining resources process Pi may request.", "word_count": 22},
        {"chunk_id": "sets_p10_c1", "material_id": "sets_demo", "material_name": "Sets.pdf", "page_number": 10, "section_heading": "Power Set", "text": "A power set of a set S is defined by P(S) = {A : A ⊆ S}. The number of elements in the power set of a finite set with n elements is 2^n.", "word_count": 33}
    ]

vstore = MaterialVectorStore("sentence-transformers/all-MiniLM-L6-v2")
vstore.index_chunks(chunks)
print(f"Successfully indexed {len(vstore.chunks)} chunks. Embeddings shape: {vstore.embeddings.shape}")


## Step 2: Query Retrieval with Cosine Similarity
Let's run semantic queries and verify top retrieved passages.


In [ ]:
query = "How do we compute the remaining need in Banker's algorithm?"
results = vstore.retrieve(query=query, top_k=3)

print(f"Query: '{query}'
")
for i, r in enumerate(results, 1):
    print(f"Rank {i} (Similarity: {r['similarity_score']:.4f}) | Doc: {r.get('material_name')} (Page {r.get('page_number')}):")
    print(f"  {r['text'][:120]}...
")


## Step 3: Evaluate Recall@K and MRR@K
We evaluate retrieval quality against benchmark queries defined in `../data/retrieval_benchmarks.json`.


In [ ]:
bench_file = "../data/retrieval_benchmarks.json"
if os.path.exists(bench_file):
    with open(bench_file, "r", encoding="utf-8") as f:
        benchmarks = json.load(f)
    
    # Run evaluation
    metrics_top3 = vstore.evaluate_retrieval(benchmarks, top_k=3)
    metrics_top5 = vstore.evaluate_retrieval(benchmarks, top_k=5)
    
    print("=== Retrieval Benchmark Results ===")
    print(f"Recall@3: {metrics_top3.get('recall@3', 0):.2%}")
    print(f"MRR@3:    {metrics_top3.get('mrr@3', 0):.4f}")
    print(f"Recall@5: {metrics_top5.get('recall@5', 0):.2%}")
    print(f"MRR@5:    {metrics_top5.get('mrr@5', 0):.4f}")


## Step 4: Verify Material-Level Isolation
A quiz for 'Deadlock.pdf' must NEVER retrieve chunks from 'Sets.pdf'.


In [ ]:
deadlock_mats = [c['material_id'] for c in chunks if 'deadlock' in c.get('material_name', '').lower()]
if deadlock_mats:
    target_mat_id = deadlock_mats[0]
    isolated_results = vstore.retrieve(
        query="What is a power set?",
        top_k=3,
        filter_material_id=target_mat_id
    )
    print(f"Retrieved {len(isolated_results)} chunks when querying 'power set' inside Deadlock material:")
    for r in isolated_results:
        print(f" - Material ID: {r['material_id']} | Text: {r['text'][:60]}...")
